In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import flexiznam as flz
import sys
from cottage_analysis.preprocessing import synchronisation
from cottage_analysis.analysis.gratings import generate_trials_df
from cottage_analysis.analysis.gratings import analyze_grating_responses

In [7]:
def find_recordings(base_dir="/Volumes/lab-znamenskiyp/home/shared/projects/"):
    base_path = Path(base_dir)
    print(f"Base path set to: {base_path}")

    project_name = "toksozi_in-vivo-BRISC"
    mouse_name = "BRAC10754.8c"
    session_name = "S20251028"
    protocol = "SFTF"

    session_path = base_path / project_name / mouse_name / session_name
    
    if not session_path.exists():
        print(f"\n[ERROR] The path does not exist:\n{session_path}")
        print("Please check your spelling and try again.")
        return None, None, None

    print(f"\nLooking for recordings in: {session_path}")
    recordings = sorted([p for p in session_path.iterdir() if p.is_dir() and p.name.startswith("R")])
    
    if not recordings:
        print("[WARNING] No recordings found starting with 'R' in this session folder.")
        return None, None, None
    
    print(f"Found {len(recordings)} recording(s):")
    for rec in recordings:
        print(f" - {rec.name}")
    flexiznam_session_id = f"{mouse_name}_{session_name}"
    
    return project_name, flexiznam_session_id, protocol


In [11]:
def run_analysis(project_name, session_id, protocol):
    if not project_name or not session_id:
        print("Invalid project or session ID. Aborting analysis.")
        return

    print(f"\n>>> Starting {protocol} analysis for {session_id}...")
    
    try:
        trials_df, dff_mean_all = analyze_grating_responses(
            project=project_name,
            session=session_id,
            protocol_base=protocol
        )
        
        print("\n>>> Analysis Complete!")
        print(f"Result: Trials DataFrame with shape {trials_df.shape}")

        return trials_df, dff_mean_all

    except Exception as e:
        print(f"\n[ERROR] Analysis failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

In [12]:
if __name__ == "__main__":
    # Find the recordings
    proj, sess_id, proto = find_recordings()
    
    # If valid recordings were found, run the analysis
    if proj and sess_id:
        run_analysis(proj, sess_id, proto)
    

Base path set to: /Volumes/lab-znamenskiyp/home/shared/projects

Looking for recordings in: /Volumes/lab-znamenskiyp/home/shared/projects/toksozi_in-vivo-BRISC/BRAC10754.8c/S20251028
Found 1 recording(s):
 - R094333_SFTF

>>> Starting SFTF analysis for BRAC10754.8c_S20251028...
Loading existing monitor frames...
Removing frames in wrong order of frame indices.
Removed 12 frames including:
0 negative diffs.
12 duplicates.
Removed 0 frames including:
0 negative diffs.
0 duplicates.
2 frames are not 0.0334 s
ImagingFrames in video: 206024
ImagingFrame triggers: 206027
Triggers - Frames: 3
FRAME NUMBER NOT CORRECT likely due to incomplete imaging volume at the end of the stack or bonsai crash.

>>> Analysis Complete!
Result: Trials DataFrame with shape (1709, 23)
